# 03｜年度滚动 ViT A2 多损失调参训练

损失函数、训练 batch 和验证选模指标是三个独立开关。Run All 会依次执行预检、2018–2025 × 3 seeds 训练、prediction 集成与完整性校验；不会调用 factor-eval。


## 步骤 1：设置本次实验参数

`LOSS_NAME`、`TRAIN_BATCH_MODE` 与 `VALIDATION_METRIC` 分别配置。仅时序换手损失因计算相邻完整截面，明确要求 `trade_date`、自然日期顺序和每个 epoch 更新一次；其他损失仍可独立搭配两种 batch。时序换手损失把前一交易日预测缓存为 stop-gradient 固定锚点，每个训练日只执行一次模型前向。


In [1]:
from pathlib import Path
import subprocess
import sys

CURRENT_DIR = Path.cwd().resolve()
VIT_DIR = CURRENT_DIR if (CURRENT_DIR / 'vit_stage07_training.py').exists() else CURRENT_DIR / 'ViT'
TRAINING_STAGE_PATH = VIT_DIR / 'vit_stage07_training.py'

# ------------------------- 实验身份 -------------------------
READ_ONLY_FOLDERS = {'baseline', 'date_batch'}
FOLDER_NAME = 'sigmoid_rank_penalty_turnover'  # 每组配置使用独立文件夹名

# ------------------------- 三个独立开关 -------------------------
LOSS_NAME = 'sigmoid_rank_ic_turnover'  # sigmoid_rank_ic_turnover / pearson_huber / smooth_l
TRAIN_BATCH_MODE = 'trade_date'             # trade_date / fixed_samples；不会随损失函数自动改变
VALIDATION_METRIC = 'daily_mean_spearman_ic'  # daily_mean_spearman_ic / daily_mean_pearson_ic

# ------------------------- 损失函数参数 -------------------------
RANK_TEMPERATURE = 0.1                     # Sigmoid / NeuralSort 软排序温度
TURNOVER_LAMBDA = 0.1                       # 时序换手惩罚 lambda，要求 >= 0
HUBER_DELTA = 1.0                          # Huber 与 Pearson-Huber 使用
SMOOTH_L1_BETA = 0.1                       # Smooth L1 二次区间宽度
HYBRID_WEIGHT = 0.5                        # Pearson-Huber 中 Pearson 项权重

# ------------------------- 其他可调超参数 -------------------------
FIXED_BATCH_SIZE = 5000                    # 仅 fixed_samples 用于训练
INFERENCE_BATCH_SIZE = 5000                # 验证数据读取与 prediction batch
LEARNING_RATE = 0.01                       # 显式参数，不由损失函数内部改写
MAX_EPOCHS = 500
PATIENCE = 10

# ------------------------- A2 调度参数 -------------------------
YEARS = '2018,2019,2020,2021,2022,2023,2024,2025'
SEEDS = '42,43,44'
DEVICE_IDS = '0,1,2,3'
TASKS_PER_GPU = 2
WORKERS_PER_TASK = 2
COMPILE_MODE = 'default'  # 动态日截面使用默认编译；禁止 reduce-overhead 的 CUDAGraph 私有池
OVERWRITE = 0  # 断点续训保持 0；明确从头覆盖同名实验时改为 1

if FOLDER_NAME in READ_ONLY_FOLDERS:
    raise ValueError('baseline 和 date_batch 是只读历史实验，请设置新的 FOLDER_NAME')
if RANK_TEMPERATURE <= 0 or TURNOVER_LAMBDA < 0 or HUBER_DELTA <= 0 or SMOOTH_L1_BETA <= 0:
    raise ValueError('温度、delta、beta 必须大于 0，TURNOVER_LAMBDA 必须大于等于 0')
if LOSS_NAME == 'sigmoid_rank_ic_turnover' and TRAIN_BATCH_MODE != 'trade_date':
    raise ValueError('sigmoid_rank_ic_turnover 必须使用 trade_date batch')
if LOSS_NAME == 'sigmoid_rank_ic_turnover' and COMPILE_MODE == 'reduce-overhead':
    raise ValueError('动态日截面禁止 reduce-overhead，请使用 COMPILE_MODE=default')
if not 0 <= HYBRID_WEIGHT <= 1:
    raise ValueError('HYBRID_WEIGHT 必须位于 [0, 1]')

A2_OPTIONS = [
    '--data-root', str(VIT_DIR / 'data'),
    '--run-name', FOLDER_NAME,
    '--device-ids', DEVICE_IDS,
    '--tasks-per-gpu', str(TASKS_PER_GPU),
    '--years', YEARS,
    '--seeds', SEEDS,
    '--train-batch-mode', TRAIN_BATCH_MODE,
    '--batch-size', str(FIXED_BATCH_SIZE),
    '--inference-batch-size', str(INFERENCE_BATCH_SIZE),
    '--loss-name', LOSS_NAME,
    '--rank-temperature', str(RANK_TEMPERATURE),
    '--turnover-lambda', str(TURNOVER_LAMBDA),
    '--huber-delta', str(HUBER_DELTA),
    '--smooth-l1-beta', str(SMOOTH_L1_BETA),
    '--hybrid-weight', str(HYBRID_WEIGHT),
    '--validation-metric', VALIDATION_METRIC,
    '--workers-per-task', str(WORKERS_PER_TASK),
    '--learning-rate', str(LEARNING_RATE),
    '--max-epochs', str(MAX_EPOCHS),
    '--patience', str(PATIENCE),
    '--compile', '1',
    '--compile-mode', COMPILE_MODE,
    '--fused-adam', '1',
    '--cuda-prefetch', '1',
    '--overwrite', str(OVERWRITE),
]

print({
    'run': FOLDER_NAME,
    '损失函数': LOSS_NAME,
    '训练batch模式': TRAIN_BATCH_MODE,
    '验证选模指标': VALIDATION_METRIC,
    'tau/turnover_lambda/delta/beta/hybrid_weight': (
        RANK_TEMPERATURE, TURNOVER_LAMBDA, HUBER_DELTA,
        SMOOTH_L1_BETA, HYBRID_WEIGHT
    ),
    '日期reshuffle': LOSS_NAME != 'sigmoid_rank_ic_turnover',
    '梯度更新': ('每epoch一次' if LOSS_NAME == 'sigmoid_rank_ic_turnover' else '每batch一次'),
    '上一日预测梯度': ('stop_gradient' if LOSS_NAME == 'sigmoid_rank_ic_turnover' else '不适用'),
    '每训练日前向次数': (1 if LOSS_NAME == 'sigmoid_rank_ic_turnover' else '按batch'),
    'compile mode': COMPILE_MODE,
    '固定batch': FIXED_BATCH_SIZE,
    'prediction batch': INFERENCE_BATCH_SIZE,
    '任务数': len(YEARS.split(',')) * len(SEEDS.split(',')),
    '每卡并发': TASKS_PER_GPU,
})


{'run': 'sigmoid_rank_penalty_turnover', '损失函数': 'sigmoid_rank_ic_turnover', '训练batch模式': 'trade_date', '验证选模指标': 'daily_mean_spearman_ic', 'tau/turnover_lambda/delta/beta/hybrid_weight': (0.1, 0.1, 1.0, 0.1, 0.5), '日期reshuffle': False, '梯度更新': '每epoch一次', '上一日预测梯度': 'stop_gradient', '每训练日前向次数': 1, 'compile mode': 'default', '固定batch': 5000, 'prediction batch': 5000, '任务数': 24, '每卡并发': 2}


## 步骤 2：数据边界、batch 统计、模型、损失函数与显存预检

预检使用当前 `LOSS_NAME × TRAIN_BATCH_MODE` 组合。时序换手损失会在第一张物理 GPU 上同时启动 `TASKS_PER_GPU` 个独立进程，每个进程覆盖最大截面在内的 12 组动态 shape，执行 stop-gradient 时序损失、反向传播和优化器更新。只有同卡并发预检通过后才进入正式训练；任何 OOM 都直接报错。


In [2]:
subprocess.run([sys.executable, str(TRAINING_STAGE_PATH), 'preflight', *A2_OPTIONS], cwd=VIT_DIR, check=True)


Python/PyTorch：3.12.4 / 2.12.1+cu130
运行名：sigmoid_rank_penalty_turnover
设备：cuda:0；可见 GPU：4
训练 batch 模式：trade_date；固定/预测 batch：5000/5000；workers/lr：2/0.01
损失函数：sigmoid_rank_ic_turnover；tau/turnover_lambda/delta/beta/hybrid_weight：0.1/0.1/1.0/0.1/0.5
验证选模指标：daily_mean_spearman_ic；reshuffle：False；梯度更新：每 epoch 一次
compile/mode/fused Adam/CUDA prefetch：True/default/True/True
GPU：NVIDIA GeForce RTX 5090，显存 31.4GB，计算能力 sm_120
   samples  label_valid  ... dataset_signature rolling_signature
0  7846954      7720274  ...      7e549f6c4204      7b816b2a7607

[1 rows x 7 columns]
   forecast_year  ... prediction_sample_count
0           2018  ...                  796559
1           2019  ...                  861870
2           2020  ...                  916587
3           2021  ...                 1027544
4           2022  ...                 1120684
5           2023  ...                 1195010
6           2024  ...                 1200893
7           2025  ...                       0

[8 rows x 21

CompletedProcess(args=['/root/miniconda3/bin/python', '/root/autodl-tmp/ViT/vit_stage07_training.py', 'preflight', '--data-root', '/root/autodl-tmp/ViT/data', '--run-name', 'sigmoid_rank_penalty_turnover', '--device-ids', '0,1,2,3', '--tasks-per-gpu', '2', '--years', '2018,2019,2020,2021,2022,2023,2024,2025', '--seeds', '42,43,44', '--train-batch-mode', 'trade_date', '--batch-size', '5000', '--inference-batch-size', '5000', '--loss-name', 'sigmoid_rank_ic_turnover', '--rank-temperature', '0.1', '--turnover-lambda', '0.1', '--huber-delta', '1.0', '--smooth-l1-beta', '0.1', '--hybrid-weight', '0.5', '--validation-metric', 'daily_mean_spearman_ic', '--workers-per-task', '2', '--learning-rate', '0.01', '--max-epochs', '500', '--patience', '10', '--compile', '1', '--compile-mode', 'default', '--fused-adam', '1', '--cuda-prefetch', '1', '--overwrite', '0'], returncode=0)

## 步骤 3：启动 8 年 × 3 seeds 正式训练

同一命令可用于 epoch 边界的断点续训。普通日截面损失按 seed 与 epoch 重排日期；时序换手损失固定按自然日期升序，前一日预测以 stop-gradient 缓存，每个训练日只前向一次，并在每个 epoch 末只执行一次优化器更新。


In [3]:
subprocess.run([sys.executable, str(TRAINING_STAGE_PATH), 'a2', *A2_OPTIONS], cwd=VIT_DIR, check=True)


A2 启动：24 个任务，GPU=(0, 1, 2, 3)，每卡并发=2，损失=sigmoid_rank_ic_turnover，batch=trade_date
启动 GPU0: 2018/seed=42 pid=884514
启动 GPU0: 2018/seed=43 pid=884515
启动 GPU1: 2018/seed=44 pid=884516
启动 GPU1: 2019/seed=42 pid=884517
启动 GPU2: 2019/seed=43 pid=884518
启动 GPU2: 2019/seed=44 pid=884519
启动 GPU3: 2020/seed=42 pid=884520
启动 GPU3: 2020/seed=43 pid=884521
完成 GPU0: 2018/seed=42 1.70 分钟
启动 GPU0: 2020/seed=44 pid=887883
完成 GPU1: 2019/seed=42 2.78 分钟
启动 GPU1: 2021/seed=42 pid=888371
完成 GPU0: 2018/seed=43 3.12 分钟
启动 GPU0: 2021/seed=43 pid=888734
完成 GPU2: 2019/seed=43 3.33 分钟
启动 GPU2: 2021/seed=44 pid=889121
完成 GPU3: 2020/seed=43 4.32 分钟
启动 GPU3: 2022/seed=42 pid=889740
完成 GPU3: 2020/seed=42 6.43 分钟
启动 GPU3: 2022/seed=43 pid=890246
完成 GPU0: 2020/seed=44 6.08 分钟
启动 GPU0: 2022/seed=44 pid=890718
完成 GPU2: 2019/seed=44 8.38 分钟
启动 GPU2: 2023/seed=42 pid=891096
完成 GPU0: 2021/seed=43 6.03 分钟
启动 GPU0: 2023/seed=43 pid=891543
完成 GPU1: 2021/seed=42 6.70 分钟
启动 GPU1: 2023/seed=44 pid=891982
完成 GPU1: 2018/seed=44 12

CompletedProcess(args=['/root/miniconda3/bin/python', '/root/autodl-tmp/ViT/vit_stage07_training.py', 'a2', '--data-root', '/root/autodl-tmp/ViT/data', '--run-name', 'sigmoid_rank_penalty_turnover', '--device-ids', '0,1,2,3', '--tasks-per-gpu', '2', '--years', '2018,2019,2020,2021,2022,2023,2024,2025', '--seeds', '42,43,44', '--train-batch-mode', 'trade_date', '--batch-size', '5000', '--inference-batch-size', '5000', '--loss-name', 'sigmoid_rank_ic_turnover', '--rank-temperature', '0.1', '--turnover-lambda', '0.1', '--huber-delta', '1.0', '--smooth-l1-beta', '0.1', '--hybrid-weight', '0.5', '--validation-metric', 'daily_mean_spearman_ic', '--workers-per-task', '2', '--learning-rate', '0.01', '--max-epochs', '500', '--patience', '10', '--compile', '1', '--compile-mode', 'default', '--fused-adam', '1', '--cuda-prefetch', '1', '--overwrite', '0'], returncode=0)

## 步骤 4：验收 checkpoint、prediction 和 T 日 Universe 键


In [4]:
subprocess.run([sys.executable, str(TRAINING_STAGE_PATH), 'validate', *A2_OPTIONS], cwd=VIT_DIR, check=True)


   forecast_year  checkpoints  prediction_samples  prediction_file       status
0           2018            3              796559             True      模型与预测完成
1           2019            3              861870             True      模型与预测完成
2           2020            3              916587             True      模型与预测完成
3           2021            3             1027544             True      模型与预测完成
4           2022            3             1120684             True      模型与预测完成
5           2023            3             1195010             True      模型与预测完成
6           2024            3             1200893             True      模型与预测完成
7           2025            3                   0            False  模型完成，暂无图片预测


CompletedProcess(args=['/root/miniconda3/bin/python', '/root/autodl-tmp/ViT/vit_stage07_training.py', 'validate', '--data-root', '/root/autodl-tmp/ViT/data', '--run-name', 'sigmoid_rank_penalty_turnover', '--device-ids', '0,1,2,3', '--tasks-per-gpu', '2', '--years', '2018,2019,2020,2021,2022,2023,2024,2025', '--seeds', '42,43,44', '--train-batch-mode', 'trade_date', '--batch-size', '5000', '--inference-batch-size', '5000', '--loss-name', 'sigmoid_rank_ic_turnover', '--rank-temperature', '0.1', '--turnover-lambda', '0.1', '--huber-delta', '1.0', '--smooth-l1-beta', '0.1', '--hybrid-weight', '0.5', '--validation-metric', 'daily_mean_spearman_ic', '--workers-per-task', '2', '--learning-rate', '0.01', '--max-epochs', '500', '--patience', '10', '--compile', '1', '--compile-mode', 'default', '--fused-adam', '1', '--cuda-prefetch', '1', '--overwrite', '0'], returncode=0)

## 步骤 5：下一步

训练验收完成后，切换到 `04_evaluate_oos_backtest.ipynb` 的 factor-eval 内核，把其中 `FOLDER_NAME` 设置为本 Notebook 的同名变量，再执行 Run All。


In [5]:
print({'训练run': FOLDER_NAME, '下一步': '在 Notebook 04 使用同一 FOLDER_NAME 执行因子评估和回测'})


{'训练run': 'sigmoid_rank_penalty_turnover', '下一步': '在 Notebook 04 使用同一 FOLDER_NAME 执行因子评估和回测'}
